In [ ]:
#!pip install -q requests pandas pyjstat

In [15]:
import pandas as pd
import requests
import time
import os
from pyjstat import pyjstat

relative_data_path = r'C:\Users\Administrator\Personal-Projects\Trade_data_Analysis\Data'

 Combined Nomenclature (CN) CODES - https://www.cbs.nl/en-gb/participants-survey/businesses/overview/international-trade-in-goods/code-lists-itgs

In [16]:
product_codes = pd.read_excel(os.path.join(relative_data_path, "Commoditycodes 2026.xlsx"), dtype={'CN2026':'str'})
country_codes = pd.read_csv(os.path.join(relative_data_path, "country_codes_V202501.csv"), dtype={'country_code':'str'})
product_codes.head()

,CN2026,SU,Description
0,01012100,p/st,Pure-bred breeding horses
1,01012910,p/st,Horses for slaughter
2,01012990,p/st,"Live horses (excl. for slaughter, pure-bred fo..."
3,01013000,p/st,Live asses
4,01019000,p/st,Live mules and hinnies


In [ ]:
country_codes.head()

In [18]:
# --- Configuration ---
DATASET = "ds-045409"
BASE_URL = f"https://ec.europa.eu/eurostat/api/comext/dissemination/statistics/1.0/data/{DATASET}"
time_period = "2025-01"
EU_countries = ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK']

In [20]:
def fetch_data_with_cn8(product_list, partner_list, batch_size=50, csv_file='monthly_data.csv'):
    file_exists = os.path.isfile(csv_file)
    flow_map = {"1": "export", "2": "import", 1: "export", 2: "import"}

    for reporter in EU_countries:
        for i in range(0, len(product_list), batch_size):
            batch = product_list[i:i + batch_size]
            params = {
                "format": "JSON",
                "lang": "EN",
                "freq": "M",
                "reporter": reporter,
                "partner": partner_list,
                "product": batch,
                "flow": ["1", "2"],
                "indicators": "VALUE_IN_EUROS",
                "TIME_PERIOD": time_period
            }

            try:
                response = requests.get(BASE_URL, params=params)

                # handle oversized payload or non-OK responses quietly
                if response.status_code == 413:
                    time.sleep(0.6)
                    continue
                if response.status_code != 200:
                    time.sleep(0.6)
                    continue

                dataset = pyjstat.Dataset.read(response.url)
                df = dataset.write('dataframe', naming='id')

                if df.empty:
                    time.sleep(0.6)
                    continue

                # keep only positive values, preserve leading zeros, map flow, merge descriptions
                df = df[df['value'] > 0]
                df['product'] = df['product'].astype(str)
                product_codes['code'] = product_codes['code'].astype(str)
                df['flow'] = df['flow'].map(flow_map).fillna(df['flow'].astype(str))
                df = df.merge(product_codes, left_on='product', right_on='code', how='left')

                df.to_csv(csv_file, mode='a', index=False, header=not file_exists)
                file_exists = True

                time.sleep(0.6)  # polite delay for the API

            except Exception:
                time.sleep(0.6)
                continue


In [21]:
## Ensure they are strings
my_product_list = product_codes['CN2026'].astype(str).tolist()
my_partner_list = country_codes['country_iso2'].dropna().astype(str).unique().tolist()


In [ ]:
fetch_data_with_cn8(my_product_list, my_partner_list)